# 03_01 · IM：可复用采样映射与插值价格序列

本 Notebook 仅生成两个数据文件，均包含时间／全合约成交量／全合约成交额三种时钟及 5／10／15 分钟对应的三档粒度：

- `im_clock_sampling_map.parquet`：采样点所在的源分钟索引、分钟内分数、价格插值左端分钟、前一个采样点位置、时间与区间活动量。
- `im_clock_sampled_prices.parquet`：同键对齐的插值价格、log 价格、区间收益、duration 和价格可用时刻。

默认以北京时间 **2026-06-01 09:30** 为训练／测试边界，训练段倒采、测试段正采，再恢复日历时间顺序。时间钟累计实际交易分钟，休市不计时；成交量／额时钟使用每分钟**所有 IM 固定月份合约**的汇总值。

训练段 $N_g=\lfloor T_\mathrm{train}/g\rfloor$，成交量／额阈值分别为训练总量除以 $N_g$，测试段沿用。每一侧输出“阈值点 + 一个方向起点”，所以 $N$ 个等时钟区间对应 $N+1$ 个价格点。每个 split 按时间排序后的第一行没有前一区间，区间字段为 null；未触达下一阈值的外端不额外制造价格点，余量记入 metadata。

`source_minute_index` 是源文件按 `bar_at` 升序后的 **0-based 行号**。`minute_fraction` 始终从该分钟开始向结束计量：0 为开始、1 为结束，倒采也不改变其方向。**在 log price 空间，以源网格上一条交易分钟 close 为左端、当前分钟 close 为右端**：
$\ell=(1-\alpha)\log C_{i-1}+\alpha\log C_i,\qquad P=\exp(\ell).$
价格左端索引 `left_close_minute_index` 与前一个采样点所在分钟 `previous_source_minute_index` 是不同概念，分别保存。源分钟时间戳与合约代码同时保留，metadata 记录源文件 SHA-256；其他分钟指标必须先对齐这个分钟网格，再使用位置索引。

相邻采样点的前／后分钟索引和分数定义了完整区间，支持一分钟内多个采样点、跨分钟、零活动分钟、午休和隔夜。`duration_seconds` 包含休市，`trading_duration_minutes` 仅包含交易分钟覆盖；`interval_volume`／`interval_money` 为区间内全合约活动量的比例分摊。

训练侧方向起点位于最后来源分钟的结束；测试侧起点位于首分钟开始，价格使用上一条交易分钟 close。跨午休／隔夜时，该价格沿用到复市首分钟开始，随后在首分钟内插值；休市本身不产生采样点。`left_close_at` 保留真实左端行情时间，`sample_at` 表示当前交易分钟内的位置，不跨 split 构造收益或区间。

价格沿用上游首尾锚定双向复权，具有全快照依赖；`available_at` 同时考虑来源分钟结束与上游复权可用时刻。本 Notebook 只生成映射和价格，不重建主力，不执行 BPV 或其他指标重采样。源网格首分钟前的 close 从本地数据库读取一次，并按首分钟的价格复权比例换算，作为最左端参考值记录在 metadata；其左端网格索引为 null，不能当成数组的 -1。


历史验证扩展位于第 7 节：三个区段分别在区段开始前校准并冻结时钟，只返回内存结果及共用分钟活动，不建立独立中间数据文件。既有 2026-06-01 锚点默认路径保留。`clock-validation-kernel` 标记单元格可独立加载；调用 `build_validation_clock_inputs(research_dir, fold_ids)` 不依赖本 Notebook 先前执行状态，也不发布任何文件。


**阶段阅读导航**：[01_01 主力连续价格](01_01_im_main_continuous.ipynb) → [02_01 BPV季节因子](02_01_im_bpv_intraday_seasonality.ipynb)（[02_02 窗口比较](02_02_im_bpv_window_comparison.ipynb)）→ [03_01 采样时钟与价格](03_01_im_clock_resampling.ipynb)（[03_02 时钟诊断](03_02_im_clock_comparison_diagnostics.ipynb)）→ [04_01 波动观测量](04_01_im_volatility_observations.ipynb) → [04_02 每日拟合](04_02_im_volatility_model_fitting.ipynb) → [04_03 下一采样步预测](04_03_im_next_step_volatility_forecasting.ipynb)／[04_04 未来15分钟预测](04_04_im_15m_volatility_forecasting.ipynb)。括号中的比较与诊断本用于查看对应阶段结果，不是下一阶段的数据生成依赖；最后两本预测相互独立。


## 1. 参数与输入／输出契约

In [1]:
import pathlib
import sys

project_markers = [".git", ".env", "config/settings.py"]
current_path = pathlib.Path.cwd().resolve()  # 当前工作目录

for candidate_root in [current_path, *current_path.parents]:
    if all((candidate_root / marker).exists() for marker in project_markers):
        sys.path.insert(0, str(candidate_root))
        break
else:
    raise RuntimeError("未找到项目根目录")

from config.settings import settings

import json
import hashlib
import numpy as np
import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq
import pyarrow.dataset as ds
import duckdb
from IPython.display import display
from config.data_contracts import (
    FUTURES_CONTRACT_CALENDAR_SCHEMA, validate_arrow_table, pandas_to_arrow, arrow_to_pandas,
)
from config.notebook_schema_browser import display_schema_metadata
from importlib import import_module

local_contract = import_module(
    "01_project_collection.JQ_strategy.financial_futures_data.financial_futures_local_contract"
)

research_dir = candidate_root / "01_project_collection/JQ_strategy/draft_experiments"
input_path = research_dir / "im_main_continuous_1m.parquet"
database_path = research_dir.parent / "financial_futures_data/data/warehouse/financial_futures.duckdb"
anchor_at = pd.Timestamp("2026-06-01 09:30:00", tz="Asia/Shanghai")
granularities = (5, 10, 15)
price_prefix = "bidirectional_adjusted_"  # "" 使用未复权价格；不改变全合约活动时钟。
clock_columns = {
    "time": "trading_minutes",
    "volume": "all_contract_volume",
    "money": "all_contract_money",
}
price_columns = ["close"]
if price_prefix not in ("", "bidirectional_adjusted_"):
    raise ValueError("本入口支持未复权或当前双向复权 close")
input_schema = pa.schema([
    pa.field("contract_code", pa.string(), False),
    pa.field("trading_date", pa.date32(), False),
    pa.field("session_number", pa.int8(), False),
    pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field(price_prefix + "close", pa.float64(), False),
    *([pa.field("close", pa.float64(), False)] if price_prefix else []),
    pa.field("price_adjustment_available_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
])
sample_key_fields = [
    pa.field("clock", pa.string(), False),
    pa.field("equivalent_minutes", pa.int16(), False),
    pa.field("split", pa.string(), False),
    pa.field("sample_number", pa.int64(), False),
]
sample_interval_fields = [
    pa.field("direction", pa.string(), False),
    pa.field("anchor_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("sample_step_from_anchor", pa.int64(), False),
    pa.field("is_anchor", pa.bool_(), False),
    pa.field("sampling_unit", pa.float64(), False),
    pa.field("sample_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("previous_sample_at", pa.timestamp("us", tz="Asia/Shanghai")),
    pa.field("has_previous_sample", pa.bool_(), False),
    pa.field("duration_seconds", pa.float64()),
    pa.field("trading_duration_minutes", pa.float64()),
    pa.field("nontrading_duration_seconds", pa.float64()),
    pa.field("interval_volume", pa.float64()),
    pa.field("interval_money", pa.float64()),
    pa.field("interval_clock_amount", pa.float64()),
    pa.field("interval_source_minute_count", pa.int32()),
]
sampling_map_schema = pa.schema([
    *sample_key_fields, *sample_interval_fields,
    pa.field("source_minute_index", pa.int64(), False),
    pa.field("minute_fraction", pa.float64(), False),
    pa.field("left_close_minute_index", pa.int64()),
    pa.field("left_close_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("source_minute_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("contract_code", pa.string(), False),
    pa.field("trading_date", pa.date32(), False),
    pa.field("session_number", pa.int8(), False),
    pa.field("previous_source_minute_index", pa.int64()),
    pa.field("previous_minute_fraction", pa.float64()),
])
sample_price_schema = pa.schema([
    *sample_key_fields, *sample_interval_fields,
    pa.field("price", pa.float64(), False),
    pa.field("log_price", pa.float64(), False),
    pa.field("log_return", pa.float64()),
    pa.field("source_minute_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("price_adjustment_available_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("available_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
])
minute_activity_schema = pa.schema([
    pa.field("bar_at", pa.timestamp("us", tz="Asia/Shanghai"), False),
    pa.field("source_minute_index", pa.int64(), False),
    pa.field("trading_date", pa.date32(), False),
    pa.field("session_number", pa.int8(), False),
    pa.field("contract_code", pa.string(), False),
    pa.field("all_contract_volume", pa.float64(), False),
    pa.field("all_contract_money", pa.float64(), False),
])
sample_keys = [field.name for field in sample_key_fields]
display(pd.DataFrame([
    {"schema": name, "field": field.name, "type": str(field.type), "nullable": field.nullable}
    for name, schema in [("input", input_schema), ("sampling_map", sampling_map_schema), ("prices", sample_price_schema), ("minute_activity", minute_activity_schema)]
    for field in schema
]))
print(sys.executable)
from IPython import get_ipython
if get_ipython() is not None:
    display_schema_metadata([FUTURES_CONTRACT_CALENDAR_SCHEMA])


,schema,field,type,nullable
0,input,contract_code,string,False
1,input,trading_date,date32[day],False
2,input,session_number,int8,False
3,input,bar_at,"timestamp[us, tz=Asia/Shanghai]",False
4,input,bidirectional_adjusted_close,double,False
5,input,close,double,False
6,input,price_adjustment_available_at,"timestamp[us, tz=Asia/Shanghai]",False
7,sampling_map,clock,string,False
8,sampling_map,equivalent_minutes,int16,False
9,sampling_map,split,string,False


E:\anaconda3\envs\latitude\python.exe


英文表名,中文表名,字段数,主键,Hive 分区,Schema 版本
dim_futures_contract_calendar,期货合约 Session 日历维度表,21,"contract_code,trading_date,session_number","exchange_code,year,month",1.2.0


## 2. 读取连续 close 与全合约分钟活动量

复用本地只读输入与日历覆盖检查。源分钟网格在全样本排序后固定，两个 split 使用同一套全局分钟索引；累计量分别从采样边界向两侧增加。

In [2]:
input_source_sha256 = hashlib.sha256(input_path.read_bytes()).hexdigest()
input_table = pq.read_table(input_path, columns=input_schema.names)
input_table = validate_arrow_table(input_table, input_schema)
minute_df = arrow_to_pandas(input_table, input_schema).rename(columns={
    price_prefix + column: "price_" + column for column in price_columns
})
minute_df["bar_at"] = pd.to_datetime(minute_df["bar_at"]).dt.tz_convert("Asia/Shanghai")
minute_df["price_adjustment_available_at"] = pd.to_datetime(minute_df["price_adjustment_available_at"]).dt.tz_convert("Asia/Shanghai")
if price_prefix == "":
    minute_df["price_adjustment_available_at"] = minute_df["bar_at"]
minute_df = minute_df.sort_values("bar_at").reset_index(drop=True)
minute_df["minute_start_at"] = minute_df["bar_at"] - pd.Timedelta(minutes=1)
if minute_df.empty or minute_df["bar_at"].duplicated().any():
    raise ValueError("连续价格必须为非空且每分钟一行")
if anchor_at.tzinfo is None:
    raise ValueError("anchor_at 必须显式带时区")
if ((minute_df["minute_start_at"] < anchor_at) & (minute_df["bar_at"] > anchor_at)).any():
    raise ValueError("训练／测试分界不能切进来源分钟，否则训练插值会引用测试段的 close")

with duckdb.connect(str(database_path), read_only=True) as connection:
    connection.execute("SET TimeZone='Asia/Shanghai'")
    # 只复核本地分钟表的物理契约，不重跑采集器的历史质量审计。
    actual_columns = connection.execute("DESCRIBE futures_minute").fetchall()
    expected_columns = local_contract.DATABASE_TABLE_COLUMNS["futures_minute"]
    normalize_type = {"TIMESTAMP WITH TIME ZONE": "TIMESTAMPTZ"}
    if [(row[0], normalize_type.get(row[1], row[1]), row[2] == "YES")
        for row in actual_columns] != list(expected_columns):
        raise ValueError("本地 futures_minute 的字段／类型与可执行契约不符")
    activity_df = connection.execute("""
        SELECT bar_at,
               sum(volume) AS all_contract_volume,
               sum(money) AS all_contract_money,
               count(*) AS active_contract_count
        FROM futures_minute
        WHERE regexp_full_match(contract_code, 'IM[0-9]{4}[.]CCFX')
          AND bar_at >= ? AND bar_at <= ?
        GROUP BY bar_at ORDER BY bar_at
    """, [minute_df["bar_at"].iloc[0].to_pydatetime(),
          minute_df["bar_at"].iloc[-1].to_pydatetime()]).fetchdf()
    # 源网格没有首分钟之前的行，只读同一首主力合约的前一条实际分钟 close 作为左端参考。
    left_boundary_row = connection.execute("""
        SELECT bar_at, close FROM futures_minute
        WHERE contract_code=? AND bar_at<?
        ORDER BY bar_at DESC LIMIT 1
    """, [str(minute_df["contract_code"].iloc[0]),
          minute_df["bar_at"].iloc[0].to_pydatetime()]).fetchone()

if left_boundary_row is None:
    raise ValueError("首分钟缺少可用的上一分钟 close，不能用 open 或自身 close 冒充")
left_boundary_at, left_boundary_raw_close = left_boundary_row
first_raw_close = float(minute_df["close"].iloc[0] if price_prefix else minute_df["price_close"].iloc[0])
if not np.isfinite([left_boundary_raw_close, first_raw_close]).all() or min(left_boundary_raw_close, first_raw_close) <= 0:
    raise ValueError("左端参考价必须有限且为正")
left_boundary_close = float(left_boundary_raw_close) * float(minute_df["price_close"].iloc[0]) / first_raw_close


# 外连接避免丢掉有全合约活动、但没有连续价格的分钟。
minute_df = minute_df.merge(activity_df, on="bar_at", how="outer", validate="one_to_one", indicator=True)
if not minute_df["_merge"].eq("both").all():
    raise ValueError("连续价格与全合约分钟集合不一致；不能静默删分钟或将缺失填零")
minute_df = minute_df.drop(columns="_merge").sort_values("bar_at").reset_index(drop=True)
# 全合约时钟要求本次输入分钟覆盖日历中的所有存续 IM 合约。
calendar_table_name = FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"table_name"].decode("utf-8")
calendar_partition_columns = FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"partition_columns"].decode("utf-8").split(",")
calendar_primary_key = FUTURES_CONTRACT_CALENDAR_SCHEMA.metadata[b"primary_key"].decode("utf-8").split(",")
calendar_columns = ["contract_code", "trading_date"]
calendar_projection_schema = pa.schema([
    FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column) for column in calendar_columns
])
calendar_dataset = ds.dataset(
    settings.futures_lake_root / "silver" / calendar_table_name,
    format="parquet",
    partitioning=ds.partitioning(pa.schema([
        FUTURES_CONTRACT_CALENDAR_SCHEMA.field(column) for column in calendar_partition_columns
    ]), flavor="hive"),
)
calendar_table = calendar_dataset.to_table(
    columns=calendar_columns,
    filter=(
        (ds.field("exchange_code") == "CCFX") & (ds.field("underlying_code") == "IM")
        & (ds.field("trading_date") >= minute_df["trading_date"].min())
        & (ds.field("trading_date") <= minute_df["trading_date"].max())
    ),
)
calendar_table = validate_arrow_table(calendar_table, calendar_projection_schema)
contract_calendar_df = arrow_to_pandas(calendar_table, calendar_projection_schema)
expected_contract_count = contract_calendar_df.groupby("trading_date", observed=True)["contract_code"].nunique()
observed_trading_dates = pd.Index(minute_df["trading_date"].unique())
if set(observed_trading_dates) != set(expected_contract_count.index):
    raise ValueError("连续价格缺少日历中的整交易日，不能压缩为休市")
minute_df["expected_contract_count"] = minute_df["trading_date"].map(expected_contract_count)
if not minute_df["active_contract_count"].eq(minute_df["expected_contract_count"]).all():
    raise ValueError("部分分钟的合约数不等于日历中的存续合约数，不能以不完整活动量采样")
numeric_columns = [*["price_" + column for column in price_columns],
                   "all_contract_volume", "all_contract_money"]
if not np.isfinite(minute_df[numeric_columns].to_numpy(dtype=np.float64)).all():
    raise ValueError("采样要求价格和活动量有限，缺失不能替代为 0")
if (minute_df[["all_contract_volume", "all_contract_money"]] < 0).any().any():
    raise ValueError("活动时钟不能为负")
if (minute_df[["price_close"]] <= 0).any().any():
    raise ValueError("插值价格必须为正")

# 本研究输入是 IM 日盘 240 个分钟；缺少整日由交易日历识别，缺少日内分钟在此拒绝。
# 固定结构用于研究前提，不能把未知缺口解释成午休。
expected_slots = [
    *pd.date_range("2000-01-01 09:31", "2000-01-01 11:30", freq="min").time,
    *pd.date_range("2000-01-01 13:01", "2000-01-01 15:00", freq="min").time,
]
slot_counts = minute_df.groupby("trading_date", observed=True)["bar_at"].agg(list)
if any([stamp.time() for stamp in stamps] != expected_slots for stamps in slot_counts):
    raise ValueError("IM 输入存在日内缺口或不同交易时段，不能将缺失分钟压缩为时间钟")
minute_df["source_minute_index"] = np.arange(len(minute_df), dtype=np.int64)
minute_df["trading_minutes"] = 1.0
minute_df["split"] = np.where(minute_df["bar_at"] <= anchor_at, "train", "test")
directed_minute_by_split = {
    "train": minute_df.loc[minute_df["split"].eq("train")].iloc[::-1].copy(),
    "test": minute_df.loc[minute_df["split"].eq("test")].copy(),
}
if any(side_df.empty for side_df in directed_minute_by_split.values()):
    raise ValueError("分界必须在现有数据覆盖范围内，且两侧均有分钟")
for split, directed_minute_df in directed_minute_by_split.items():
    directed_minute_df["direction"] = "backward" if split == "train" else "forward"
    for clock, column in clock_columns.items():
        directed_minute_df[clock + "_cusum"] = directed_minute_df[column].to_numpy(dtype=np.float64).cumsum()

display(minute_df.groupby("split", sort=False).agg(
    minute_count=("bar_at", "size"),
    first_minute=("bar_at", "min"),
    last_minute=("bar_at", "max"),
    min_contracts=("active_contract_count", "min"),
    max_contracts=("active_contract_count", "max"),
))


,minute_count,first_minute,last_minute,min_contracts,max_contracts
split,,,,,
train,223440,2022-07-25 09:31:00+08:00,2026-05-29 15:00:00+08:00,4,4
test,15360,2026-06-01 09:31:00+08:00,2026-08-28 15:00:00+08:00,4,4


## 3. 累计量除以阈值，批量定位采样点

令 $q_i=C_i/x$。对整数目标 $k=1,\dots,\lfloor q_\mathrm{last}\rfloor$，用 `searchsorted` 找到沿采样方向第一个达到目标的正活动分钟；方向比例 $u=(k-q_{i-1})/(q_i-q_{i-1})$。正采的分钟分数为 $u$，倒采为 $1-u$；**两种方向保存的 fraction 都按分钟开始→结束计量**。

例如还差 3 手、该分钟有 5 手：正采保存 3/5；倒采从分钟末端已消耗 3/5，保存的正向时间位置为 2/5。价格均在“上一条交易分钟 log close→当前分钟 log close”之间按保存分数插值。

方向起点 step=0，训练位于最近分钟末端，测试位于首分钟开始。多个目标落在同一分钟时保留多个点；零成交分钟不产生新阈值，其时间覆盖仍由相邻位置表示。

In [3]:
def locate_sampling_points(directed_minute_df, clock, sampling_unit):
    """由单侧累计量批量求出全部阈值点与方向起点的全局分钟索引、分钟内分数。"""
    if clock not in clock_columns or not np.isfinite(sampling_unit) or sampling_unit <= 0:
        raise ValueError("采样时钟须有效，阈值须为有限正数")
    if directed_minute_df.empty:
        raise ValueError("采样侧不能没有分钟")
    clock_increment = directed_minute_df[clock_columns[clock]].to_numpy(dtype=np.float64)
    clock_quotient = directed_minute_df[clock + "_cusum"].to_numpy(dtype=np.float64) / sampling_unit
    if not np.isfinite(clock_quotient).all() or (clock_increment < 0).any():
        raise ValueError("累计活动量必须有限且非负")
    # 只消除机器精度级别的整数边界误差，保证校准后的第 N 个阈值不会丢失。
    tolerance = 32 * np.finfo(np.float64).eps * max(1.0, clock_quotient[-1])
    rounded_quotient = np.rint(clock_quotient)
    clock_quotient = np.where(
        np.abs(clock_quotient - rounded_quotient) <= tolerance, rounded_quotient, clock_quotient
    )
    quotient_before = np.r_[0.0, clock_quotient[:-1]]
    quotient_increment = clock_quotient - quotient_before
    positive_minute_index = np.flatnonzero(clock_increment > 0)
    if (quotient_increment[positive_minute_index] <= 0).any():
        raise ValueError("浮点累计量精度不足以区分正增量")
    sample_step = np.arange(1, int(np.floor(clock_quotient[-1])) + 1, dtype=np.int64)
    # 同一来源分钟可以命中多个阈值；零活动分钟不会被选作阈值落点。
    hit_minute_index = positive_minute_index[np.searchsorted(
        clock_quotient[positive_minute_index], sample_step, side="left"
    )]
    directed_fraction = (
        (sample_step - quotient_before[hit_minute_index]) / quotient_increment[hit_minute_index]
    ).clip(0, 1)
    backward = directed_minute_df["direction"].iloc[0] == "backward"
    minute_fraction = 1 - directed_fraction if backward else directed_fraction
    sampling_locations_df = pd.DataFrame({
        "sample_step_from_anchor": np.r_[0, sample_step],
        "source_minute_index": directed_minute_df["source_minute_index"].to_numpy()[
            np.r_[0, hit_minute_index]
        ],
        "minute_fraction": np.r_[1.0 if backward else 0.0, minute_fraction],
    })
    return sampling_locations_df.sort_values(
        "sample_step_from_anchor", ascending=not backward
    ).reset_index(drop=True)


## 4. 校准阈值，构造采样位置和价格

只用训练段校准交易量／额阈值。每个组合恢复时间顺序后，用前一个采样点的位置计算 duration 和区间活动量；收益对齐到区间右端的价格点。分钟索引／分数仅保存供后续使用，本次不计算其他指标的重采样值。

In [4]:
def build_clock_samples(minute_df, directed_minute_by_split, anchor_at, left_boundary_close, left_boundary_at):
    """按一个冻结锚点生成全局分钟索引不变的采样映射、价格和校准结果。"""
    training_minute_df = directed_minute_by_split["train"]
    source_close = minute_df["price_close"].to_numpy(dtype=np.float64)
    source_left_close = np.r_[left_boundary_close, source_close[:-1]]
    source_log_close = np.log(source_close)
    source_left_log_close = np.log(source_left_close)
    source_start_us = pd.DatetimeIndex(minute_df["minute_start_at"]).tz_convert("UTC").as_unit("us").asi8
    source_end_us = pd.DatetimeIndex(minute_df["bar_at"]).tz_convert("UTC").as_unit("us").asi8
    price_adjustment_us = pd.DatetimeIndex(minute_df["price_adjustment_available_at"]).tz_convert("UTC").as_unit("us").asi8
    volume_prefix = np.r_[0.0, minute_df["all_contract_volume"].to_numpy(dtype=float).cumsum()]
    money_prefix = np.r_[0.0, minute_df["all_contract_money"].to_numpy(dtype=float).cumsum()]
    calibration_rows, sampling_map_frames, sample_price_frames = [], [], []

    for equivalent_minutes in granularities:
        training_interval_count = int(training_minute_df["time_cusum"].iloc[-1] // equivalent_minutes)
        if training_interval_count < 1:
            raise ValueError("训练段不足一个时间采样单位")
        for clock, column in clock_columns.items():
            training_clock_total = float(training_minute_df[clock + "_cusum"].iloc[-1])
            sampling_unit = float(equivalent_minutes) if clock == "time" else training_clock_total / training_interval_count
            if sampling_unit <= 0:
                raise ValueError(f"训练段 {clock} 总量为 0，无法校准阈值")
            for split, directed_minute_df in directed_minute_by_split.items():
                # 累计量及其商保留在内存，文件只保存可复用的采样位置。
                directed_minute_df[f"{clock}_quotient_{equivalent_minutes}m"] = (
                    directed_minute_df[clock + "_cusum"] / sampling_unit
                )
                sampling_map_df = locate_sampling_points(directed_minute_df, clock, sampling_unit)
                source_index = sampling_map_df["source_minute_index"].to_numpy(dtype=np.int64)
                fraction = sampling_map_df["minute_fraction"].to_numpy(dtype=float)
                sampling_map_df["sample_number"] = np.arange(len(sampling_map_df))
                sampling_map_df["clock"] = clock
                sampling_map_df["equivalent_minutes"] = equivalent_minutes
                sampling_map_df["split"] = split
                sampling_map_df["direction"] = directed_minute_df["direction"].iloc[0]
                sampling_map_df["anchor_at"] = anchor_at
                sampling_map_df["is_anchor"] = sampling_map_df["sample_step_from_anchor"].eq(0)
                sampling_map_df["sampling_unit"] = sampling_unit
                sample_us = source_start_us[source_index] + np.rint(fraction * 60_000_000).astype(np.int64)
                sampling_map_df["sample_at"] = pd.to_datetime(sample_us, unit="us", utc=True).tz_convert("Asia/Shanghai")
                sampling_map_df["previous_sample_at"] = sampling_map_df["sample_at"].shift(1)
                sampling_map_df["has_previous_sample"] = sampling_map_df["sample_number"].gt(0)
                sampling_map_df["previous_source_minute_index"] = pd.array(source_index, dtype="int64[pyarrow]")
                sampling_map_df["previous_source_minute_index"] = sampling_map_df["previous_source_minute_index"].shift(1)
                sampling_map_df["previous_minute_fraction"] = sampling_map_df["minute_fraction"].shift(1)
                # 交易分钟坐标剔除了所有休市间隔；相邻位置之差就是覆盖分钟数。
                sampling_map_df["trading_duration_minutes"] = np.r_[np.nan, np.diff(source_index) + np.diff(fraction)]
                sampling_map_df["duration_seconds"] = sampling_map_df["sample_at"].diff().dt.total_seconds()
                sampling_map_df["nontrading_duration_seconds"] = (
                    sampling_map_df["duration_seconds"] - 60 * sampling_map_df["trading_duration_minutes"]
                ).clip(lower=0)
                for field, prefix, source_column in [
                    ("interval_volume", volume_prefix, "all_contract_volume"),
                    ("interval_money", money_prefix, "all_contract_money"),
                ]:
                    source_values = minute_df[source_column].to_numpy(dtype=float)
                    left_index, right_index = source_index[:-1], source_index[1:]
                    left_fraction, right_fraction = fraction[:-1], fraction[1:]
                    interval_total = np.where(
                        left_index == right_index,
                        source_values[left_index] * (right_fraction - left_fraction),
                        prefix[right_index] - prefix[left_index + 1]
                        + source_values[left_index] * (1 - left_fraction)
                        + source_values[right_index] * right_fraction,
                    )
                    sampling_map_df[field] = np.r_[np.nan, interval_total]
                sampling_map_df["interval_clock_amount"] = sampling_map_df[{
                    "time": "trading_duration_minutes", "volume": "interval_volume", "money": "interval_money"
                }[clock]]
                covered_minute_count = (
                    source_index[1:] - source_index[:-1] + 1
                    - (fraction[:-1] == 1).astype(int) - (fraction[1:] == 0).astype(int)
                )
                sampling_map_df["interval_source_minute_count"] = pd.array(
                    [None, *covered_minute_count], dtype="int32[pyarrow]"
                )
                for source_column in ["contract_code", "trading_date", "session_number"]:
                    sampling_map_df[source_column] = minute_df[source_column].iloc[source_index].reset_index(drop=True)
                sampling_map_df["left_close_minute_index"] = pd.array(
                    np.where(source_index > 0, source_index - 1, None), dtype="int64[pyarrow]"
                )
                left_close_us = source_end_us[np.maximum(source_index - 1, 0)].copy()
                left_close_us[source_index == 0] = pd.Timestamp(left_boundary_at).tz_convert("UTC").as_unit("us").asm8.view("i8")
                sampling_map_df["left_close_at"] = pd.to_datetime(
                    left_close_us, unit="us", utc=True
                ).tz_convert("Asia/Shanghai")
                sampling_map_df["source_minute_at"] = pd.to_datetime(
                    source_end_us[source_index], unit="us", utc=True
                ).tz_convert("Asia/Shanghai")

                sample_price_df = sampling_map_df[
                    [field.name for field in sample_key_fields + sample_interval_fields]
                ].copy()
                sample_price_df["log_price"] = source_left_log_close[source_index] + fraction * (
                    source_log_close[source_index] - source_left_log_close[source_index]
                )
                sample_price_df["price"] = np.exp(sample_price_df["log_price"])
                sample_price_df["log_return"] = sample_price_df["log_price"].diff()
                sample_price_df["source_minute_at"] = sampling_map_df["source_minute_at"]
                sample_price_df["price_adjustment_available_at"] = pd.to_datetime(
                    price_adjustment_us[source_index], unit="us", utc=True
                ).tz_convert("Asia/Shanghai")
                sample_price_df["available_at"] = pd.to_datetime(
                    np.maximum(source_end_us[source_index], price_adjustment_us[source_index]),
                    unit="us", utc=True,
                ).tz_convert("Asia/Shanghai")
                sampling_map_frames.append(sampling_map_df[sampling_map_schema.names])
                sample_price_frames.append(sample_price_df[sample_price_schema.names])
                interval_count = len(sampling_map_df) - 1
                side_total = float(directed_minute_df[clock + "_cusum"].iloc[-1])
                calibration_rows.append({
                    "clock": clock, "equivalent_minutes": equivalent_minutes, "split": split,
                    "sampling_unit": sampling_unit, "training_interval_count": training_interval_count,
                    "side_clock_total": side_total, "interval_count": interval_count,
                    "point_count": len(sampling_map_df),
                    "unreached_outer_clock_amount": max(0.0, side_total - interval_count * sampling_unit),
                })

    sampling_map_df = pd.concat(sampling_map_frames, ignore_index=True).sort_values(sample_keys).reset_index(drop=True)
    sample_price_df = pd.concat(sample_price_frames, ignore_index=True).sort_values(sample_keys).reset_index(drop=True)
    calibration_df = pd.DataFrame(calibration_rows)
    return sampling_map_df, sample_price_df, calibration_df


def publish_clock_samples(output_dir, sampling_map_df, sample_price_df, calibration_df, anchor_at, reference_checks, extra_metadata=None):
    """将一个锚点的同身份映射与价格发布到明确指定的研究目录。"""
    # 同一映射身份和源快照写入两份文件的 metadata，校准不再单独输出 JSON。
    input_sha256 = hashlib.sha256(input_path.read_bytes()).hexdigest()
    if input_sha256 != input_source_sha256:
        raise RuntimeError("执行期间上游分钟源文件发生变化，停止发布，避免不同快照混用")
    activity_sha256 = hashlib.sha256(
        minute_df[["all_contract_volume", "all_contract_money"]].to_numpy(dtype="<f8").tobytes()
    ).hexdigest()
    sampling_metadata = {
        "format_version": "2.0.0",
        "anchor_at": anchor_at.isoformat(),
        "granularities": list(granularities),
        "source_file": str(input_path),
        "source_sha256": input_sha256,
        "source_minute_count": len(minute_df),
        "source_order": "bar_at ascending; source_minute_index is zero based",
        "source_key": ["contract_code", "bar_at"],
        "activity_sha256": activity_sha256,
        "activity_scope": "all IM fixed-month contracts; volume in lots; money in CNY",
        "database_path": str(database_path),
        "price_prefix": price_prefix,
        "price_adjustment_available_at": minute_df["price_adjustment_available_at"].max().isoformat(),
        "price_adjustment_scope": "whole-snapshot retrospective dual endpoint adjustment" if price_prefix else "raw prices",
        "price_interpolation": "log_price=(1-fraction)*log(previous trading minute close)+fraction*log(current close); price=exp(log_price)",
        "left_boundary": {"contract_code": str(minute_df["contract_code"].iloc[0]), "bar_at": pd.Timestamp(left_boundary_at).isoformat(), "raw_close": float(left_boundary_raw_close), "adjusted_close": float(left_boundary_close)},
        "downstream_use": "indices and fractions retained only; no downstream metric resampling in this run",
        "interval_geometry": "overlap of [previous_index+previous_fraction, current_index+fraction] with each [minute_index,minute_index+1]",
        "duration_seconds": "calendar elapsed seconds including recess",
        "trading_duration_minutes": "sum of covered minute fractions; recess excluded",
        "reference_point_policy": "one directional origin per split; no previous interval for first chronological point",
        "primary_key": sample_keys,
        "calibration": calibration_df.to_dict(orient="records"),
        "reference_checks": reference_checks,
    }
    sampling_metadata.update(extra_metadata or {})
    sampling_map_id = hashlib.sha256(
        json.dumps(sampling_metadata, sort_keys=True, ensure_ascii=False).encode("utf-8")
    ).hexdigest()
    common_metadata = {
        b"sampling_map_id": sampling_map_id.encode(),
        b"sampling_metadata": json.dumps(sampling_metadata, ensure_ascii=False).encode("utf-8"),
        b"primary_key": ",".join(sample_keys).encode(),
    }
    output_tables = [
        ("im_clock_sampling_map.parquet", pandas_to_arrow(sampling_map_df, sampling_map_schema)),
        ("im_clock_sampled_prices.parquet", pandas_to_arrow(sample_price_df, sample_price_schema)),
    ]
    output_dir.mkdir(parents=True, exist_ok=True)
    # 两个临时文件均完成 schema 与逐值复读后，再安装；不在日常 Notebook 中放历史文件删除逻辑。
    pending_paths = []
    for filename, output_table in output_tables:
        output_table = output_table.replace_schema_metadata(common_metadata)
        output_path = output_dir / filename
        temporary_path = output_path.with_suffix(".parquet.tmp")
        pq.write_table(output_table, temporary_path, compression="zstd", row_group_size=25000)
        reread_table = pq.ParquetFile(temporary_path).read()
        assert reread_table.schema.equals(output_table.schema, check_metadata=True)
        assert reread_table.equals(output_table)
        pending_paths.append((temporary_path, output_path))
    for temporary_path, output_path in pending_paths:
        temporary_path.replace(output_path)
        print(f"{output_path.name}: {len(sampling_map_df):,} 个价格点")

    return sampling_map_id


sampling_map_df, sample_price_df, calibration_df = build_clock_samples(
    minute_df, directed_minute_by_split, anchor_at, left_boundary_close, left_boundary_at,
)
# 第 5 节的独立价格复算使用原分钟数组。
source_close = minute_df["price_close"].to_numpy(dtype=np.float64)
source_log_close = np.log(source_close)
source_left_log_close = np.log(np.r_[left_boundary_close, source_close[:-1]])
display(calibration_df)
display(sampling_map_df.head())


,clock,equivalent_minutes,split,sampling_unit,training_interval_count,side_clock_total,interval_count,point_count,unreached_outer_clock_amount
0,time,5,train,5.000000e+00,44688,2.234400e+05,44688,44689,0.000000e+00
1,time,5,test,5.000000e+00,44688,1.536000e+04,3072,3073,0.000000e+00
2,volume,5,train,3.296650e+03,44688,1.473207e+08,44688,44689,0.000000e+00
3,volume,5,test,3.296650e+03,44688,1.836528e+07,5570,5571,2.937276e+03
4,money,5,train,4.192761e+09,44688,1.873661e+14,44688,44689,0.000000e+00
5,money,5,test,4.192761e+09,44688,2.838853e+13,6770,6771,3.539750e+09
6,time,10,train,1.000000e+01,22344,2.234400e+05,22344,22345,0.000000e+00
7,time,10,test,1.000000e+01,22344,1.536000e+04,1536,1537,0.000000e+00
8,volume,10,train,6.593300e+03,22344,1.473207e+08,22344,22345,0.000000e+00
9,volume,10,test,6.593300e+03,22344,1.836528e+07,2785,2786,2.937276e+03


,clock,equivalent_minutes,split,sample_number,direction,anchor_at,sample_step_from_anchor,is_anchor,sampling_unit,sample_at,previous_sample_at,has_previous_sample,duration_seconds,trading_duration_minutes,nontrading_duration_seconds,interval_volume,interval_money,interval_clock_amount,interval_source_minute_count,source_minute_index,minute_fraction,left_close_minute_index,left_close_at,source_minute_at,contract_code,trading_date,session_number,previous_source_minute_index,previous_minute_fraction
0,money,5,test,0,forward,2026-06-01 09:30:00+08:00,0,True,4.192761e+09,2026-06-01 09:30:00+08:00,NaT,False,NaN,NaN,NaN,NaN,NaN,NaN,<NA>,223440,0.000000,223439,2026-05-29 15:00:00+08:00,2026-06-01 09:31:00+08:00,IM2606.CCFX,2026-06-01,1,<NA>,NaN
1,money,5,test,1,forward,2026-06-01 09:30:00+08:00,1,False,4.192761e+09,2026-06-01 09:30:19.963319+08:00,2026-06-01 09:30:00+08:00,True,19.963319,0.332722,0.000000e+00,2528.354415,4.192761e+09,4.192761e+09,1,223440,0.332722,223439,2026-05-29 15:00:00+08:00,2026-06-01 09:31:00+08:00,IM2606.CCFX,2026-06-01,1,223440,0.000000
2,money,5,test,2,forward,2026-06-01 09:30:00+08:00,2,False,4.192761e+09,2026-06-01 09:30:39.926639+08:00,2026-06-01 09:30:19.963319+08:00,True,19.963320,0.332722,5.001290e-07,2528.354415,4.192761e+09,4.192761e+09,1,223440,0.665444,223439,2026-05-29 15:00:00+08:00,2026-06-01 09:31:00+08:00,IM2606.CCFX,2026-06-01,1,223440,0.332722
3,money,5,test,3,forward,2026-06-01 09:30:00+08:00,3,False,4.192761e+09,2026-06-01 09:30:59.889958+08:00,2026-06-01 09:30:39.926639+08:00,True,19.963319,0.332722,0.000000e+00,2528.354415,4.192761e+09,4.192761e+09,1,223440,0.998166,223439,2026-05-29 15:00:00+08:00,2026-06-01 09:31:00+08:00,IM2606.CCFX,2026-06-01,1,223440,0.665444
4,money,5,test,4,forward,2026-06-01 09:30:00+08:00,4,False,4.192761e+09,2026-06-01 09:31:33.025402+08:00,2026-06-01 09:30:59.889958+08:00,True,33.135444,0.552257,8.012472e-08,2535.976654,4.192761e+09,4.192761e+09,2,223441,0.550423,223440,2026-06-01 09:31:00+08:00,2026-06-01 09:32:00+08:00,IM2606.CCFX,2026-06-01,1,223440,0.998166


## 5. 验证位置、价格与时间区间

检查训练区间数、两表主键对齐、阈值触达、独立慢速参考、close→close 的 log 插值和区间边界。

In [5]:
# 两个文件一一对齐；首个参考点没有前一区间，其余每个区间触达一个阈值。
pd.testing.assert_frame_equal(sampling_map_df[sample_keys], sample_price_df[sample_keys])
assert not sampling_map_df.duplicated(sample_keys).any()
assert sampling_map_df["minute_fraction"].between(0, 1).all()
has_previous = sampling_map_df["has_previous_sample"]
assert sample_price_df.loc[~has_previous, "log_return"].isna().all()
assert sampling_map_df.loc[~has_previous, "previous_source_minute_index"].isna().all()
np.testing.assert_allclose(
    sampling_map_df.loc[has_previous, "interval_clock_amount"],
    sampling_map_df.loc[has_previous, "sampling_unit"], rtol=2e-9, atol=1e-8,
)
for row in calibration_df.itertuples(index=False):
    selected_df = sampling_map_df.loc[
        sampling_map_df["clock"].eq(row.clock)
        & sampling_map_df["equivalent_minutes"].eq(row.equivalent_minutes)
        & sampling_map_df["split"].eq(row.split)
    ]
    assert len(selected_df) == row.interval_count + 1
    if row.split == "train":
        assert row.interval_count == row.training_interval_count
        assert (selected_df["source_minute_at"] <= anchor_at).all()
    else:
        assert (selected_df["sample_at"] >= anchor_at).all()
    assert selected_df["sample_at"].is_monotonic_increasing
    assert not selected_df["sample_at"].duplicated().any()
    assert selected_df["is_anchor"].sum() == 1
assert (sampling_map_df.loc[has_previous, "trading_duration_minutes"] > 0).all()
assert (sample_price_df["available_at"] >= sample_price_df["sample_at"]).all()
assert (sample_price_df["available_at"] >= sample_price_df["price_adjustment_available_at"]).all()
source_index = sampling_map_df["source_minute_index"].to_numpy(dtype=int)
fraction = sampling_map_df["minute_fraction"].to_numpy(dtype=float)
np.testing.assert_allclose(
    sample_price_df["price"],
    np.exp(source_left_log_close[source_index] + fraction * (source_log_close[source_index] - source_left_log_close[source_index])),
    rtol=0, atol=0,
)

# 独立慢速阈值参考：覆盖双向、零量、单分钟多个点、恰好触达和未触达。
rng = np.random.default_rng(20260906)
reference_checks = 0
for case_number in range(5):
    amounts = np.array([0, 2, 13, 0, 5, 0], dtype=float) if case_number == 0 else rng.integers(0, 18, 6).astype(float)
    for backward in [False, True]:
        synthetic_df = pd.DataFrame({
            "source_minute_index": np.arange(6), "all_contract_volume": amounts,
            "direction": "backward" if backward else "forward",
        })
        if backward:
            synthetic_df = synthetic_df.iloc[::-1].copy()
        synthetic_df["volume_cusum"] = synthetic_df["all_contract_volume"].cumsum()
        located_df = locate_sampling_points(synthetic_df, "volume", 5.0)
        reference_rows = [(int(synthetic_df["source_minute_index"].iloc[0]), 1.0 if backward else 0.0)]
        remaining = 5.0
        for minute in synthetic_df.itertuples(index=False):
            consumed = 0.0
            while consumed < minute.all_contract_volume - 1e-10:
                taken = min(minute.all_contract_volume - consumed, remaining)
                consumed += taken
                remaining -= taken
                if remaining < 1e-10:
                    fraction = consumed / minute.all_contract_volume
                    reference_rows.append((minute.source_minute_index, 1 - fraction if backward else fraction))
                    remaining = 5.0
        if backward:
            reference_rows.reverse()
        np.testing.assert_array_equal(located_df["source_minute_index"], [row[0] for row in reference_rows])
        np.testing.assert_allclose(located_df["minute_fraction"], [row[1] for row in reference_rows], atol=1e-12)
        reference_checks += 1
zero_df = pd.DataFrame({
    "source_minute_index": [0, 1], "all_contract_volume": [0., 0.],
    "volume_cusum": [0., 0.], "direction": ["forward", "forward"],
})
assert len(locate_sampling_points(zero_df, "volume", 5.0)) == 1

# 价格空间的手工核对：100→121，分数 1/2 时 log 线性插值得到 110。
np.testing.assert_allclose(np.exp(np.log(100.) + .5 * (np.log(121.) - np.log(100.))), 110.)
print(f"验证通过：两表同键、训练等数、双向阈值定位（{reference_checks} 组独立参考）、close→close 的 log 插值与区间时长；未执行下游指标重采样。")


验证通过：两表同键、训练等数、双向阈值定位（10 组独立参考）、close→close 的 log 插值与区间时长；未执行下游指标重采样。


## 6. 仅保存采样映射与价格两个文件

用 `clock`、`equivalent_minutes`、`split` 筛选。两份文件共用映射身份与键，校准、源文件摘要、最左侧参考价及余量全部写入 metadata；累计量和商只保留在内存。

**供下游读取的区间含义**：从前一个位置 $(i,a)$ 到当前 $(j,b)$，分钟 $m$ 的覆盖比例为
$$w_m=\max(0,\min(j+b,m+1)-\max(i+a,m)).$$
这些边界足以让下游将分钟 BPV 等指标按 $\sum w_m z_m/\sum w_m$ 对齐到当前价格点。其他分钟表必须先按时间戳及合约对齐源网格；缺失值如何处理由对应下游任务决定。本次不执行这项计算。

In [6]:
sampling_map_id = publish_clock_samples(
    research_dir, sampling_map_df, sample_price_df, calibration_df, anchor_at, reference_checks,
)
display(calibration_df.pivot(index=["equivalent_minutes", "clock"], columns="split", values="point_count"))


im_clock_sampling_map.parquet: 274,056 个价格点
im_clock_sampled_prices.parquet: 274,056 个价格点


split                      test  train
equivalent_minutes clock              
5                  money   6771  44689
                   time    3073  44689
                   volume  5571  44689
10                 money   3386  22345
                   time    1537  22345
                   volume  2786  22345
15                 money   2257  14897
                   time    1025  14897
                   volume  1857  14897

## 7. 历史验证多锚点与共用分钟活动（内存接口）

历史采样属于低成本上游环节，直接向 04 返回内存中的映射、价格和活动表。此节不写 Parquet、模型对象或一次性脚本，也不改写既有最终采样文件。

| fold_id | 锚点（上海时间） | 测试结束日 | 处理 |
|---|---|---|---|
| validation_2025h1 | 2025-02-05 09:30 | 2025-06-30 | 区段开始前全部可用分钟校准；内存生成 |
| validation_2025h2 | 2025-07-01 09:30 | 2025-12-31 | 区段开始前全部可用分钟校准；内存生成 |
| validation_2026h1 | 2026-01-05 09:30 | 2026-05-29 | 区段开始前全部可用分钟校准；内存生成 |
| final_2026 | 2026-06-01 09:30 | 2026-08-28 | 只读取原目录两个既有 Parquet |

`clock-validation-kernel` 单元格只定义 `build_validation_clock_inputs`。下游按此 tag 加载后，可直接调用，不需先运行前面的单元格。函数在独立命名空间中显式复用本书的配置、只读源加载、阈值定位和采样构造代码；不执行发布环节，因此不会因 Notebook 执行顺序而隐式写文件。

返回值包含 `activity_df`（开篇 `minute_activity_schema` 对应的七个字段）、`activity_metadata`（真实来源与活动 SHA），以及 `folds[fold_id]` 中的 `map_df`、`price_df`、`sampling_metadata`、`sampling_map_id`。历史映射保持全量源分钟的全局零基索引；快照复权的历史可用性局限仍如实保留。


In [ ]:
def build_validation_clock_inputs(research_dir, fold_ids):
    """只读本地源并在内存返回指定区段；复用本书代码，不执行任何发布单元格。"""
    import ast
    import hashlib
    import json
    import pathlib

    research_dir = pathlib.Path(research_dir).resolve()
    fold_ids = list(fold_ids)
    fold_definitions = {
        "validation_2025h1": ("2025-02-05", "2025-06-30"),
        "validation_2025h2": ("2025-07-01", "2025-12-31"),
        "validation_2026h1": ("2026-01-05", "2026-05-29"),
        "final_2026": ("2026-06-01", "2026-08-28"),
    }
    if not fold_ids or len(set(fold_ids)) != len(fold_ids) or set(fold_ids) - set(fold_definitions):
        raise ValueError("fold_ids 必须是非空、不重复的已定义区段列表")
    notebook_path = research_dir / "03_01_im_clock_resampling.ipynb"
    notebook = json.loads(notebook_path.read_text(encoding="utf-8"))
    cells_by_id = {cell["id"]: cell for cell in notebook["cells"]}
    namespace = {"__name__": "im_clock_validation", "display": lambda *args, **kwargs: None}
    # 固定引用四个拥有业务实现的 cell；隔离命名空间避免外部 kernel 变量干扰。
    for cell_id in ["sampling-cell-2", "sampling-cell-4", "sampling-cell-6", "sampling-cell-8"]:
        module = ast.parse("".join(cells_by_id[cell_id]["source"]))
        if cell_id == "sampling-cell-2":
            # 开篇配置运行到首次展示之前；其后的 schema 浏览器属于交互展示。
            first_display = next(i for i, node in enumerate(module.body) if
                isinstance(node, ast.Expr) and isinstance(node.value, ast.Call)
                and isinstance(node.value.func, ast.Name) and node.value.func.id == "display")
            module.body = module.body[:first_display]
        elif cell_id == "sampling-cell-8":
            module.body = [node for node in module.body if
                isinstance(node, ast.FunctionDef) and node.name == "build_clock_samples"]
        else:
            module.body = [node for node in module.body if not (
                isinstance(node, ast.Expr) and isinstance(node.value, ast.Call)
                and isinstance(node.value.func, ast.Name) and node.value.func.id == "display"
            )]
        exec(compile(module, f"{notebook_path.name}:{cell_id}", "exec"), namespace)
        if cell_id == "sampling-cell-2":
            namespace["research_dir"] = research_dir
            namespace["input_path"] = research_dir / "im_main_continuous_1m.parquet"
            namespace["database_path"] = research_dir.parent / "financial_futures_data/data/warehouse/financial_futures.duckdb"
            namespace["display"] = lambda *args, **kwargs: None
    # 独立双向阈值参考在本次调用实际执行，metadata 的 reference_checks 不伪造。
    reference_source = "".join(cells_by_id["sampling-cell-10"]["source"])
    reference_module = ast.parse(reference_source[reference_source.index("rng = np.random.default_rng"):])
    reference_module.body = [node for node in reference_module.body if not (
        isinstance(node, ast.Expr) and isinstance(node.value, ast.Call)
        and isinstance(node.value.func, ast.Name) and node.value.func.id == "print"
    )]
    exec(compile(reference_module, "clock_threshold_reference", "exec"), namespace)
    np, pd, pq = namespace["np"], namespace["pd"], namespace["pq"]
    minute_df = namespace["minute_df"]
    activity_df = minute_df[namespace["minute_activity_schema"].names].copy()
    # 即使不落盘，也以显式 Arrow 契约校验返回的活动列。
    namespace["pandas_to_arrow"](activity_df, namespace["minute_activity_schema"])
    activity_sha256 = hashlib.sha256(
        activity_df[["all_contract_volume", "all_contract_money"]].to_numpy(dtype="<f8").tobytes()
    ).hexdigest()
    activity_metadata = {
        "source_file": str(namespace["input_path"]),
        "source_sha256": namespace["input_source_sha256"],
        "activity_sha256": activity_sha256,
    }
    # 复用发布器内的纯 metadata 表达式；不执行发布器及其文件 I/O。
    construction_module = ast.parse("".join(cells_by_id["sampling-cell-8"]["source"]))
    publisher = next(node for node in construction_module.body if
        isinstance(node, ast.FunctionDef) and node.name == "publish_clock_samples")
    metadata_assignment = next(node for node in publisher.body if
        isinstance(node, ast.Assign) and any(isinstance(target, ast.Name)
            and target.id == "sampling_metadata" for target in node.targets))
    metadata_module = ast.Module(body=[metadata_assignment], type_ignores=[])
    fold_inputs = {}
    for fold_id in fold_ids:
        fold_start, fold_end = fold_definitions[fold_id]
        fold_anchor_at = pd.Timestamp(fold_start + " 09:30:00", tz="Asia/Shanghai")
        fold_end_at = pd.Timestamp(fold_end + " 15:00:00", tz="Asia/Shanghai")
        if fold_id == "final_2026":
            map_table = pq.read_table(research_dir / "im_clock_sampling_map.parquet")
            price_table = pq.read_table(research_dir / "im_clock_sampled_prices.parquet")
            map_metadata, price_metadata = map_table.schema.metadata, price_table.schema.metadata
            if map_metadata[b"sampling_map_id"] != price_metadata[b"sampling_map_id"]:
                raise ValueError("既有最终映射和价格的采样身份不一致")
            sampling_metadata = json.loads(map_metadata[b"sampling_metadata"])
            if sampling_metadata["source_sha256"] != activity_metadata["source_sha256"]:
                raise ValueError("既有最终采样引用的连续价格快照与当前源不一致")
            if sampling_metadata["activity_sha256"] != activity_sha256:
                raise ValueError("既有最终采样引用的活动快照与当前活动不一致")
            if sampling_metadata["anchor_at"] != fold_anchor_at.isoformat():
                raise ValueError("既有最终采样锚点不符合 final_2026 定义")
            map_df = namespace["arrow_to_pandas"](
                namespace["validate_arrow_table"](map_table, namespace["sampling_map_schema"]),
                namespace["sampling_map_schema"],
            )
            price_df = namespace["arrow_to_pandas"](
                namespace["validate_arrow_table"](price_table, namespace["sample_price_schema"]),
                namespace["sample_price_schema"],
            )
            within_end = pd.to_datetime(map_df["source_minute_at"]) <= fold_end_at
            map_df = map_df.loc[within_end].reset_index(drop=True)
            price_df = price_df.loc[within_end].reset_index(drop=True)
            sampling_map_id = map_metadata[b"sampling_map_id"].decode()
        else:
            directed_minute_by_split = {
                "train": minute_df.loc[minute_df["bar_at"] <= fold_anchor_at].iloc[::-1].copy(),
                "test": minute_df.loc[(minute_df["minute_start_at"] >= fold_anchor_at)
                    & (minute_df["bar_at"] <= fold_end_at)].copy(),
            }
            if any(frame.empty for frame in directed_minute_by_split.values()):
                raise ValueError(f"{fold_id}: 两侧必须有可用分钟")
            for split, directed_minute_df in directed_minute_by_split.items():
                directed_minute_df["direction"] = "backward" if split == "train" else "forward"
                for clock, column in namespace["clock_columns"].items():
                    directed_minute_df[clock + "_cusum"] = directed_minute_df[column].to_numpy(dtype=np.float64).cumsum()
            map_df, price_df, calibration_df = namespace["build_clock_samples"](
                minute_df, directed_minute_by_split, fold_anchor_at,
                namespace["left_boundary_close"], namespace["left_boundary_at"],
            )
            namespace.update({"input_sha256": activity_metadata["source_sha256"],
                "activity_sha256": activity_sha256, "anchor_at": fold_anchor_at,
                "calibration_df": calibration_df})
            exec(compile(metadata_module, "sampling_metadata", "exec"), namespace)
            sampling_metadata = dict(namespace["sampling_metadata"])
            sampling_metadata.update({"fold_id": fold_id,
                "evaluation_start_at": fold_anchor_at.isoformat(),
                "evaluation_end_at": fold_end_at.isoformat(),
                "calibration_scope": "all available source minutes strictly before evaluation anchor; frozen within fold"})
            sampling_map_id = hashlib.sha256(
                json.dumps(sampling_metadata, sort_keys=True, ensure_ascii=False).encode("utf-8")
            ).hexdigest()
            namespace["pandas_to_arrow"](map_df, namespace["sampling_map_schema"])
            namespace["pandas_to_arrow"](price_df, namespace["sample_price_schema"])
        sample_keys = namespace["sample_keys"]
        pd.testing.assert_frame_equal(map_df[sample_keys], price_df[sample_keys])
        assert not map_df.duplicated(sample_keys).any()
        valid_interval = map_df["has_previous_sample"].to_numpy(dtype=bool)
        np.testing.assert_allclose(map_df.loc[valid_interval, "interval_clock_amount"].to_numpy(dtype=float),
            map_df.loc[valid_interval, "sampling_unit"].to_numpy(dtype=float), rtol=2e-9, atol=1e-8)
        assert (map_df.loc[valid_interval, "trading_duration_minutes"] > 0).all()
        assert price_df.loc[~valid_interval, "log_return"].isna().all()
        source_index = map_df["source_minute_index"].to_numpy(dtype=np.int64)
        np.testing.assert_array_equal(pd.to_datetime(map_df["source_minute_at"]).to_numpy(),
            pd.to_datetime(minute_df["bar_at"].iloc[source_index]).to_numpy())
        for (_, _, split), group_df in map_df.groupby(["clock", "equivalent_minutes", "split"], observed=True):
            assert pd.to_datetime(group_df["sample_at"]).is_monotonic_increasing
            assert not group_df["sample_at"].duplicated().any()
            assert group_df["is_anchor"].sum() == 1
            if split == "test":
                assert (pd.to_datetime(group_df["sample_at"]) >= fold_anchor_at).all()
                assert (pd.to_datetime(group_df["source_minute_at"]) <= fold_end_at).all()
            else:
                assert (pd.to_datetime(group_df["source_minute_at"]) <= fold_anchor_at).all()
        fold_inputs[fold_id] = {"map_df": map_df, "price_df": price_df,
            "sampling_metadata": sampling_metadata, "sampling_map_id": sampling_map_id}
    if hashlib.sha256(namespace["input_path"].read_bytes()).hexdigest() != activity_metadata["source_sha256"]:
        raise RuntimeError("执行期间上游分钟快照发生变化")
    return {"activity_df": activity_df, "activity_metadata": activity_metadata, "folds": fold_inputs}


In [ ]:
# 此单元格只展示统计；下游也可以独立加载上一单元格的函数后直接调用。
validation_inputs = build_validation_clock_inputs(research_dir, [
    "validation_2025h1", "validation_2025h2", "validation_2026h1", "final_2026",
])
validation_summary_rows = []
for fold_id, fold_inputs in validation_inputs["folds"].items():
    for (clock, equivalent_minutes, split), group_df in fold_inputs["map_df"].groupby(
        ["clock", "equivalent_minutes", "split"], observed=True,
    ):
        validation_summary_rows.append({"fold_id": fold_id, "clock": clock,
            "equivalent_minutes": equivalent_minutes, "split": split, "point_count": len(group_df)})
validation_summary_df = pd.DataFrame(validation_summary_rows)
display(validation_summary_df.pivot(index=["fold_id", "equivalent_minutes", "clock"], columns="split", values="point_count"))
print(f"共用分钟活动：{len(validation_inputs['activity_df']):,} 行；历史映射与活动仅保存在内存。")
